# PrexSyn Synthesizability Analysis

**Research question:** PrexSyn guarantees Enamine synthesizability for every returned molecule
(via `stack_size == 1` internal filter). What fraction of these guaranteed-synthesizable
molecules does AiZynthFinder independently recognise?

**Pipeline:**
1. Sample a large batch from PrexSyn (256 samples × 100 seeds)
2. Deduplicate and merge with existing baseline variants
3. Score everything with AiZynthFinder (depth 6, Enamine stock)
4. Analyse recognition rate by molecular complexity, quality bin, and property

**Key context:**
- AiZynthFinder stock = Enamine building blocks (same source as PrexSyn)
- AiZynthFinder templates = USPTO reactions (different from Enamine reaction templates)
- Any gap between 100% (PrexSyn guarantee) and the observed rate reflects
  reaction-template mismatch, not building-block mismatch

In [ ]:
import sys, json, time, urllib.request
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed, TimeoutError as FutureTimeoutError
from concurrent.futures.process import BrokenProcessPool

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors
from tqdm.notebook import tqdm

ROOT = Path('.').resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.evaluation.synth_parallel import worker_init, score_one

# ── Config ────────────────────────────────────────────────────────────────────
GEN_DIR    = ROOT / 'data' / 'generation_stratified'
OUT_DIR    = GEN_DIR / 'prexsyn_synth_analysis'
OUT_DIR.mkdir(parents=True, exist_ok=True)

AIZYNTHFINDER_CONFIG = ROOT / 'data' / 'aizynthfinder' / 'config.yml'
NPZ_PATH             = GEN_DIR / 'chembl_features_stratified.npz'
SEEDS_JSON           = GEN_DIR / 'seeds_for_methods_stratified.json'
BASELINE_CSV         = GEN_DIR / 'baseline' / 'baseline_scores.csv'

PREXSYN_URL  = 'http://100.65.172.100:8011/sample'
NUM_SAMPLES  = 256    # samples per seed per call
MAX_DEPTH    = 6
TIME_LIMIT   = 120
N_WORKERS    = 10

LARGE_BATCH_JSON = OUT_DIR / 'prexsyn_large_batch.json'
SCORED_CSV       = OUT_DIR / 'prexsyn_large_batch_scored.csv'
SYNTH_CKPT       = OUT_DIR / f'synth_depth{MAX_DEPTH}.json'

print(f'ROOT          : {ROOT}')
print(f'NPZ           : {"OK" if NPZ_PATH.exists() else "MISSING"}')
print(f'AiZynth cfg   : {"OK" if AIZYNTHFINDER_CONFIG.exists() else "MISSING"}')
print(f'Baseline CSV  : {"OK" if BASELINE_CSV.exists() else "MISSING"}')
print(f'Samples/seed  : {NUM_SAMPLES}')
print(f'AiZynth depth : {MAX_DEPTH}')
print(f'Output dir    : {OUT_DIR}')

---
## Stage 1 — Sample large PrexSyn batch

Call PrexSyn with 256 samples per seed (vs 64 in the original baseline).  
Results are cached to `prexsyn_large_batch.json` — re-running this cell is safe.

In [ ]:
def call_prexsyn(payload: dict, url: str, retries: int = 3) -> list[str]:
    for attempt in range(retries):
        try:
            req  = urllib.request.Request(
                url, data=json.dumps(payload).encode(),
                headers={'Content-Type': 'application/json'},
            )
            resp = json.loads(urllib.request.urlopen(req, timeout=180).read())
            return resp.get('generated_smiles', [])
        except Exception as e:
            wait = 2 ** attempt
            print(f'  attempt {attempt+1}/{retries} failed: {e}  (retry in {wait}s)')
            time.sleep(wait)
    return []

# Load features
npz         = np.load(NPZ_PATH, allow_pickle=True)
smiles_arr  = npz['smiles']
rdkit_names = npz['rdkit_desc_names'].tolist()
seeds_meta  = json.load(open(SEEDS_JSON))
spec_to_bin = {s['spec_smiles']: s['quality_bin'] for s in seeds_meta}
n_seeds     = len(smiles_arr)

# Resume from cache
if LARGE_BATCH_JSON.exists():
    results = json.load(open(LARGE_BATCH_JSON))
    done    = {r['spec_smiles'] for r in results}
    print(f'Resumed: {len(done)}/{n_seeds} seeds already sampled')
else:
    results = []
    done    = set()

pending_idx = [i for i in range(n_seeds) if str(smiles_arr[i]) not in done]
print(f'Remaining: {len(pending_idx)} seeds to sample ({NUM_SAMPLES} samples each)')

for i in tqdm(pending_idx, desc='PrexSyn sampling'):
    spec = str(smiles_arr[i])
    payload = {
        'ecfp4':             npz['ecfp4'][i].tolist(),
        'fcfp4':             npz['fcfp4'][i].tolist(),
        'rdkit_desc_values': npz['rdkit_desc_values'][i].tolist(),
        'rdkit_desc_names':  rdkit_names,
        'brics_fps':         npz['brics_fps'][i].tolist(),
        'brics_exists':      npz['brics_exists'][i].tolist(),
        'source_smiles':     spec,
        'num_samples':       NUM_SAMPLES,
    }
    smiles_out = call_prexsyn(payload, PREXSYN_URL)
    results.append({
        'spec_smiles':      spec,
        'quality_bin':      spec_to_bin.get(spec, 'unknown'),
        'generated_smiles': smiles_out,
        'n_generated':      len(smiles_out),
    })
    # Save checkpoint after every seed
    LARGE_BATCH_JSON.write_text(json.dumps(results, indent=2))

total_gen = sum(r['n_generated'] for r in results)
print(f'\nTotal generated SMILES : {total_gen:,}')
print(f'Saved to               : {LARGE_BATCH_JSON}')

---
## Stage 2 — Build scored DataFrame

Flatten all generated SMILES, deduplicate, compute molecular properties,
and merge with existing baseline variants.

In [ ]:
def mol_props(smi: str) -> dict:
    """Compute basic molecular properties. Returns None-filled dict on failure."""
    try:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            return {}
        return {
            'mw':        Descriptors.ExactMolWt(mol),
            'hba':       rdMolDescriptors.CalcNumHBA(mol),
            'hbd':       rdMolDescriptors.CalcNumHBD(mol),
            'rotbonds':  rdMolDescriptors.CalcNumRotatableBonds(mol),
            'rings':     rdMolDescriptors.CalcNumRings(mol),
            'arom_rings': rdMolDescriptors.CalcNumAromaticRings(mol),
            'heavy_atoms': mol.GetNumHeavyAtoms(),
        }
    except Exception:
        return {}

# Flatten into rows
rows = []
seen = set()
for r in results:
    for smi in r['generated_smiles']:
        if smi in seen:
            continue
        seen.add(smi)
        row = {'variant_smiles': smi,
               'spec_smiles':    r['spec_smiles'],
               'quality_bin':    r['quality_bin'],
               'source':         'prexsyn_large_batch'}
        row.update(mol_props(smi))
        rows.append(row)

df_new = pd.DataFrame(rows)
print(f'Large batch: {len(df_new):,} unique variant SMILES')

# Merge with existing baseline (may contain some of same molecules)
if BASELINE_CSV.exists():
    df_base = pd.read_csv(BASELINE_CSV)[['variant_smiles', 'spec_smiles']].copy()
    df_base['source'] = 'baseline'
    df_base['quality_bin'] = df_base['spec_smiles'].map(spec_to_bin)
    for smi in df_base['variant_smiles']:
        if smi not in seen:
            seen.add(smi)
            row = {'variant_smiles': smi,
                   'spec_smiles':    df_base.loc[df_base['variant_smiles']==smi, 'spec_smiles'].iloc[0],
                   'quality_bin':    df_base.loc[df_base['variant_smiles']==smi, 'quality_bin'].iloc[0],
                   'source':         'baseline'}
            row.update(mol_props(smi))
            rows.append(row)

df_all = pd.DataFrame(rows).drop_duplicates('variant_smiles').reset_index(drop=True)
print(f'Combined (large batch + baseline): {len(df_all):,} unique variants')
print(f'Quality bin distribution:')
print(df_all['quality_bin'].value_counts().sort_index())

---
## Stage 3 — AiZynthFinder scoring

Score all unique PrexSyn-generated variants. Checkpoint resumes safely.

In [ ]:
assert AIZYNTHFINDER_CONFIG.exists(), f'Missing: {AIZYNTHFINDER_CONFIG}'

# Load checkpoint
synth: dict[str, bool] = {}
if SYNTH_CKPT.exists():
    synth = json.load(open(SYNTH_CKPT))
    print(f'Resumed: {len(synth):,} already scored')

all_smiles = df_all['variant_smiles'].dropna().tolist()
pending    = [s for s in all_smiles if s not in synth]
print(f'To score: {len(pending):,} / {len(all_smiles):,}')

if pending:
    with ProcessPoolExecutor(
        max_workers=N_WORKERS,
        initializer=worker_init,
        initargs=(str(AIZYNTHFINDER_CONFIG), MAX_DEPTH, TIME_LIMIT),
        max_tasks_per_child=None,
    ) as pool:
        futures = {pool.submit(score_one, s): s for s in pending}
        with tqdm(total=len(pending), desc=f'AiZynth depth={MAX_DEPTH}', unit='mol') as pbar:
            for fut in as_completed(futures):
                smi = futures[fut]
                try:
                    _, solved = fut.result(timeout=TIME_LIMIT + 30)
                except Exception:
                    solved = False
                synth[smi] = solved
                pbar.update(1)
    SYNTH_CKPT.write_text(json.dumps(synth))

n_solved = sum(synth.values())
print(f'\nSynthesizable: {n_solved:,} / {len(synth):,}  ({100*n_solved/max(len(synth),1):.1f}%)')

# Write synth column back to df
df_all[f'is_synth_d{MAX_DEPTH}'] = df_all['variant_smiles'].map(synth).fillna(False)
df_all.to_csv(SCORED_CSV, index=False)
print(f'Saved: {SCORED_CSV}')

---
## Stage 4 — Analysis

### 4a: Overall AiZynthFinder recognition rate of PrexSyn outputs

In [ ]:
synth_col = f'is_synth_d{MAX_DEPTH}'
df_all    = pd.read_csv(SCORED_CSV)   # reload in case kernel restarted

total  = len(df_all)
n_synth = int(df_all[synth_col].sum())
rate    = n_synth / total * 100

print('=' * 55)
print('PrexSyn → AiZynthFinder Recognition Rate')
print('=' * 55)
print(f'Total unique PrexSyn variants : {total:>8,}')
print(f'AiZynthFinder synthesizable   : {n_synth:>8,}  ({rate:.1f}%)')
print(f'Not recognised                : {total-n_synth:>8,}  ({100-rate:.1f}%)')
print()
print('Interpretation:')
print(f'  PrexSyn guarantees 100% synthesizability via Enamine templates.')
print(f'  AiZynthFinder (USPTO templates + Enamine stock) recognises {rate:.1f}%.')
print(f'  The {100-rate:.1f}% gap = reaction-template mismatch, not building-block mismatch.')
print('=' * 55)

### 4b: Recognition rate by quality bin

In [ ]:
BINS = ['<0.5', '0.5-0.7', '0.7-0.85', '0.85-1.0']

print(f'Recognition rate by quality bin (depth {MAX_DEPTH}):')
print(f'{"Bin":<10}  {"Total":>7}  {"Synth":>7}  {"Rate %":>7}')
print('-' * 38)
for b in BINS:
    sub  = df_all[df_all['quality_bin'] == b]
    n    = len(sub)
    s    = int(sub[synth_col].sum())
    r    = s / n * 100 if n > 0 else float('nan')
    print(f'{b:<10}  {n:>7,}  {s:>7,}  {r:>7.1f}%')
sub_all = df_all
print(f'{"Overall":<10}  {len(sub_all):>7,}  {int(sub_all[synth_col].sum()):>7,}  '
      f'{sub_all[synth_col].mean()*100:>7.1f}%')

### 4c: Recognition rate by molecular complexity

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

props = [
    ('mw',        'Molecular Weight (Da)',  [0, 200, 300, 400, 500, 700]),
    ('rings',     'Number of Rings',        [0, 1, 2, 3, 4, 5, 10]),
    ('heavy_atoms','Heavy Atom Count',      [0, 15, 20, 25, 30, 35, 50]),
]

for ax, (col, label, bins) in zip(axes, props):
    sub = df_all[df_all[col].notna()].copy()
    sub['bin_label'] = pd.cut(sub[col], bins=bins, right=False)
    grp  = sub.groupby('bin_label', observed=True)[synth_col].agg(['mean', 'count'])
    grp['mean'] *= 100

    xs = range(len(grp))
    bars = ax.bar(xs, grp['mean'], color='steelblue', alpha=0.75, edgecolor='white')
    for bar, (_, row) in zip(bars, grp.iterrows()):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.5,
                f'{row["mean"]:.0f}%\n(n={int(row["count"])})',
                ha='center', va='bottom', fontsize=7)

    ax.set_xticks(list(xs))
    ax.set_xticklabels([str(i) for i in grp.index], rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('AiZynthFinder recognition rate (%)')
    ax.set_xlabel(label)
    ax.set_ylim(0, min(100, grp['mean'].max() * 1.3 + 5))
    ax.grid(True, axis='y', alpha=0.3)

fig.suptitle(
    f'Figure: AiZynthFinder Recognition Rate of PrexSyn Outputs by Molecular Complexity\n'
    f'(PrexSyn guarantees 100% Enamine synthesizability; AiZynthFinder uses USPTO templates)',
    fontweight='bold', y=1.02
)
plt.tight_layout()
out = OUT_DIR / 'fig_recognition_by_complexity.png'
plt.savefig(out, bbox_inches='tight', dpi=150)
plt.show()
print(f'Saved -> {out}')

### 4d: Per-seed recognition rate distribution

In [ ]:
per_seed = (
    df_all.groupby('spec_smiles')[synth_col]
    .agg(total='count', synth='sum')
    .assign(rate=lambda x: x['synth'] / x['total'] * 100)
    .reset_index()
)
per_seed['quality_bin'] = per_seed['spec_smiles'].map(spec_to_bin)

print('Per-seed recognition rate summary:')
print(f'  Mean   : {per_seed["rate"].mean():.1f}%')
print(f'  Median : {per_seed["rate"].median():.1f}%')
print(f'  Std    : {per_seed["rate"].std():.1f}%')
print(f'  Min    : {per_seed["rate"].min():.1f}%  (worst seed)')
print(f'  Max    : {per_seed["rate"].max():.1f}%  (best seed)')
print()

# Seeds with 0% recognition
zero_seeds = per_seed[per_seed['rate'] == 0]
print(f'Seeds with 0% AiZynthFinder recognition: {len(zero_seeds)}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram of per-seed rates
axes[0].hist(per_seed['rate'], bins=20, color='steelblue', alpha=0.75, edgecolor='white')
axes[0].axvline(per_seed['rate'].median(), color='red', lw=1.5, ls='--',
                label=f'Median = {per_seed["rate"].median():.1f}%')
axes[0].set_xlabel('AiZynthFinder recognition rate per seed (%)')
axes[0].set_ylabel('Number of seeds')
axes[0].set_title('Distribution of per-seed recognition rates', fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, axis='y', alpha=0.3)

# By quality bin
bin_order = ['<0.5', '0.5-0.7', '0.7-0.85', '0.85-1.0']
colors    = ['#e74c3c', '#e67e22', '#27ae60', '#2980b9']
data_bp   = [per_seed[per_seed['quality_bin'] == b]['rate'].dropna().values for b in bin_order]
bp = axes[1].boxplot(data_bp, patch_artist=True, medianprops=dict(color='black', lw=2))
for patch, col in zip(bp['boxes'], colors):
    patch.set_facecolor(col)
    patch.set_alpha(0.7)
axes[1].set_xticklabels(bin_order)
axes[1].set_xlabel('PrexSyn quality bin (Tanimoto to ChEMBL ref)')
axes[1].set_ylabel('AiZynthFinder recognition rate (%)')
axes[1].set_title('Recognition rate by quality bin', fontweight='bold')
axes[1].grid(True, axis='y', alpha=0.3)

fig.suptitle('Per-seed AiZynthFinder Recognition of PrexSyn Outputs', fontweight='bold', y=1.02)
plt.tight_layout()
out = OUT_DIR / 'fig_per_seed_recognition.png'
plt.savefig(out, bbox_inches='tight', dpi=150)
plt.show()
print(f'Saved -> {out}')

---
## Stage 5 — Key findings summary

In [ ]:
total   = len(df_all)
n_synth = int(df_all[synth_col].sum())
rate    = n_synth / total * 100

print('=' * 65)
print('SUMMARY: PrexSyn Synthesizability vs AiZynthFinder Recognition')
print('=' * 65)
print()
print(f'1. ENAMINE GUARANTEE (PrexSyn internal)')
print(f'   All {total:,} returned variants have a valid Enamine synthesis route.')
print(f'   Guaranteed rate: 100%  (stack_size == 1 filter)')
print()
print(f'2. AIZYNTHFINDER RECOGNITION (USPTO templates + Enamine stock)')
print(f'   Recognised: {n_synth:,} / {total:,}  ({rate:.1f}%)')
print(f'   Not recognised: {total-n_synth:,}  ({100-rate:.1f}%)')
print()
print(f'3. INTERPRETATION')
print(f'   The {100-rate:.1f}% gap is not a synthesizability failure.')
print(f'   It reflects reaction-template mismatch:')
print(f'     PrexSyn  uses Enamine-specific reaction templates')
print(f'     AiZynth  uses USPTO-trained retrosynthetic templates')
print(f'   Same building blocks (Enamine), different reaction vocabulary.')
print()
print(f'4. IMPLICATION FOR PAPER')
print(f'   AiZynthFinder provides a unified, method-agnostic evaluation')
print(f'   framework. For PrexSyn outputs, the {rate:.0f}% pass rate is a')
print(f'   conservative lower bound — not an estimate of true synthesizability.')
print(f'   For modification methods (CReM, mmpdb, JT-VAE, LibINVENT), which')
print(f'   have no synthesis guarantee, the pass rate is a genuine estimate.')
print('=' * 65)